In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
def plot_predicted_vs_measured(measured,predicted,dataset,plot=True):
    
    r2 = r2_score(measured, predicted)
    mae = mean_absolute_error(measured, predicted)
    rmse = mean_squared_error(measured, predicted) ** 0.5

    if plot: # Scatter plot: measured vs predicted on test set
        plt.figure(figsize=(6, 6))
        plt.scatter(measured, predicted, alpha=0.6)
        plt.xlabel("Measured power")
        plt.ylabel("Predicted power")
        plt.title(f"{dataset} model: measured vs predicted (test set)")

        min_val = min(measured.min(), predicted.min())
        max_val = max(measured.max(), predicted.max())
        plt.plot([min_val, max_val], [min_val, max_val])

        plt.show()

    
        print("Wind model")
        print("R²:", r2)
        print("MAE:", mae)
        print("RMSE:", rmse)
    
    return r2

In [ ]:
def Sarimax(Type,file_path,endo_name,exo_name,granularity=10,order = (1,0,1),seasonal_order = (1,1,1),gap = ["2025-06-14 00:00:00","2025-06-21 00:00:00"],save_new_file = False,filename="",plot = True):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes
    Type is either "Wind" or "Solar" """


    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts") # Set datetime as index
    df = df.loc[df.index >"2025-05-14 07:00:00"] # start from when our power data starts


    df[exo_name] = df[exo_name].interpolate() # REMOVE ANY NANS

    df_modelled = df.copy()
    
    if Type == "Wind":
        # Wind power is proportional to the cubic wind speed
        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan

        df["active"] = (df[exo_name]>= 3).astype(int)
        df["w2"] = df[exo_name]**2
        df["w3"] = df[exo_name]**3
        
        endo = df[endo_name]
        exo = df[[exo_name,"active","w3"]]
        print("Preparing Model for Wind Turbine")

    elif Type == "Solar":
        df.loc[df[exo_name] < 1, endo_name] = 0 # set night time to zero
        df_modelled.loc[df[exo_name] < 1, endo_name] = 0 # set night time to zero

        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        
        df = df.loc[df[exo_name] >= 1] # remove night time to reduce how much it gets skewed
        

        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan
        endo = df[endo_name]
        exo = df[exo_name] 
        print("Preparing Model for PV unit")


    if gap_measured.isna().sum().astype(int) != 0: 
        print("Gap contains NaNs so the model can not be verified. Retry with different gap.")
        return 

    fraction_dayvsnight = df.size/df_modelled.size
    seasonality = 24*60/granularity*fraction_dayvsnight ####### Takes the average time between the time today and the same time tomorrow (complex since we removed nighttime values)
    seasonal_order_full = (*seasonal_order, int(seasonality))

    model = SARIMAX(
        endo,
        exog=exo,
        order=order,
        seasonal_order=seasonal_order_full,  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()

    endo_modelled = results.get_prediction().predicted_mean # USES KALMAN SMOOTHING ( includes both past and future values to do "predictions")
    df["endo_filled"] = df[endo_name]
    df.loc[endo.isna(),"endo_filled"] = endo_modelled.loc[endo.isna()]



    # results.plot_diagnostics() # in case we want to see built in diagnostics 


    if Type == "Wind":
        df_modelled[endo_name + "modelled"] = df["endo_filled"] # Set df_modelled equal to the results! 

    elif Type == "Solar":
        df_modelled.loc[df_modelled[exo_name] >= 1, endo_name +"modelled"] = df["endo_filled"] # redo the zero rows that were removed because irradiance was zero
        df_modelled.loc[df_modelled[exo_name] < 1, endo_name+ "modelled"] = 0 # set everything at night to 0
    

    if save_new_file:
        file_name = f'{file_path[:-4]}_{filename}_{granularity}min__SARIMAX_{order}{seasonal_order}.csv'
        df_modelled.to_csv(file_name, index=True)

    
    gap_predicted = df_modelled.loc[(gap[0] < df_modelled.index) & (df_modelled.index < gap[1]),endo_name + "modelled"]

    return plot_predicted_vs_measured(gap_measured,gap_predicted,endo_name,plot)



In [ ]:
combined_data = str(ROOT / "data/combined_data_DMI_gen.csv")

SOLAR Most effective has been: 

for 60min resolution: 

(002,000) r2: 0.900

for 30min: 

(001,000) r2: 0.865


for 10min: 

(001,000) r2: 0.799


In [ ]:
Sarimax("Solar",combined_data,"B715","Global Irradiance",granularity = 60,order = (1,0,0),seasonal_order = (0,0,0))